In [1]:
# -*- coding: utf-8 -*-

# Phase 1 — 전처리와 피처 엔지니어링 (v13)

`유연오더_가상데이터_v13.xlsx` 12,000건 → 분석 테이블.

**v3 대비 달라진 점**

| | v2/v3 로그 | v4 |
|---|---|---|
| 시트 | `유연오더_등록로그` (헤더 2행) | `콜등록이력` (헤더 1행) |
| 시간창 | `상차_허용시작/종료/허용범위_분` 3필드 | `시간창_분` 1필드 |
| 하차마감 | `하차_절대마감` | **없음** (운송여유 계산 불가) |
| 자동조정 | `자동조정_허용` | **없음** |
| 거리 | 없음 → 별도 추정 필요 | `참조_노선`에 `거리km`·`표준소요_h`·`톨비` |
| 결과 | **없음** | `최종체결운임`·`결과`·`배차소요_분`·`인상횟수` |

마지막 줄이 핵심이다. v4에는 **경매 결과**가 있어서 "조건이 결과를 얼마나 바꾸는가"를
파생변수로 만들 수 있다. v2에서는 불가능했던 것들이다.

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

SRC = Path(os.getenv("DATA_DIR", ".")).expanduser().resolve()
IN_XLSX = SRC / os.getenv("DATA_FILE", "유연오더_가상데이터_v13.xlsx")
OUT_DIR = SRC / "out"
OUT_DIR.mkdir(exist_ok=True)

if not IN_XLSX.exists():
    raise FileNotFoundError(f"입력 파일 없음: {IN_XLSX}  (.env의 DATA_DIR 확인)")

print(f"[경로] {IN_XLSX}")

[경로] /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/가상데이터-최종.xlsx


## 1-0. 로딩 — 시트 5개를 한 번에 읽는다

v4는 단일 시트가 아니다. 콜·제안·노선·기준운임·알선소를 각각 읽되
**`화주프로파일`은 생성 파라미터 정답지라 분석 테이블에 병합하지 않는다.**
세그먼트 결과를 사후 검증할 때만 따로 꺼내 쓴다.

In [3]:
xl = pd.ExcelFile(IN_XLSX)
print(f"[시트] {xl.sheet_names}")

df = pd.read_excel(xl, "콜등록이력")
prop = pd.read_excel(xl, "개입제안로그")
route = pd.read_excel(xl, "참조_노선")
broker = pd.read_excel(xl, "알선소마스터")

print(f"[load] 콜 {len(df)}행 {df.shape[1]}컬럼 / 제안 {len(prop)}행 / 노선 {len(route)}")

# 노선 정보 병합 — v2에서 좌표로 추정하던 거리가 v4에는 실측값으로 들어 있다
df = df.merge(route, on="노선ID", how="left")
# 알선소 속성 병합 (구분=주선사/운송사, 관할권역)
df = df.merge(
    broker[["알선소ID", "구분", "관할권역"]].rename(
        columns={"구분": "알선소_구분", "관할권역": "알선소_관할권역"}),
    on="알선소ID", how="left")

assert df["거리km"].notna().all(), "노선 병합 실패"
assert df["알선소_구분"].notna().all(), "알선소 병합 실패"

[시트] ['_README', '검증리포트', '콜등록이력', '개입제안로그', '화주프로파일', '알선소마스터', '운송인마스터', '운송인이력', '참조_노선', '참조_기준운임']


[load] 콜 12000행 39컬럼 / 제안 9324행 / 노선 12


## 1-1. 타입 변환과 파생 시간 변수

v4는 등록일시·상차희망이 이미 완전한 datetime이다. v2처럼 `MM-DD HH:MM` 문자열에
연도를 붙이거나 자정 넘김을 보정할 필요가 없다.

시간창은 `시간창_분` 하나뿐이라 **상차희망을 중심으로 대칭이라고 가정**하고
허용 구간을 복원한다. 생성기의 가용 판정이 창의 폭에만 의존하므로 이 가정은 무해하다.

In [4]:
df["등록일시"] = pd.to_datetime(df["등록일시"])
df["상차희망"] = pd.to_datetime(df["상차희망"])

df["상차_허용시작"] = df["상차희망"] - pd.to_timedelta(df["시간창_분"] / 2, unit="m")
df["상차_허용종료"] = df["상차희망"] + pd.to_timedelta(df["시간창_분"] / 2, unit="m")

df["요일"] = df["등록일시"].dt.weekday          # 0=월
df["시간대"] = df["등록일시"].dt.hour
df["등록일"] = df["등록일시"].dt.normalize()
df["상차요일"] = df["상차희망"].dt.weekday
df["주말상차"] = (df["상차요일"] >= 5).astype(int)

# 리드타임은 v4에 이미 있다. 재계산해 정합성만 확인한다.
_lead_calc = (df["상차희망"] - df["등록일시"]).dt.total_seconds() / 3600
print(f"[정합성] 리드타임 재계산 최대오차 {(_lead_calc - df['리드타임_h']).abs().max():.3f}h")

# 개입 구간 분리 — v13 생성기가 기록한 기준일 이전=학습용, 이후=개입 검증용
_generation_meta = SRC / "생성_메타_v13.json"
if not _generation_meta.exists():
    raise FileNotFoundError(f"생성 메타 없음: {_generation_meta} — v13 생성기를 먼저 실행하세요")
CUT = pd.Timestamp(json.loads(_generation_meta.read_text(encoding="utf-8"))["cut_date"])
print(f"[구간 기준] {CUT:%Y-%m-%d} 이전=개입전 / 이후=개입후")
df["구간"] = np.where(df["등록일시"] < CUT, "개입전", "개입후")

[정합성] 리드타임 재계산 최대오차 0.050h


## 1-2. 유연성 종합지수 — 가중치 재배분

v2 공식은 `time .40 / veh .15 / cargo .25 / auth .20`이었다.
v4에는 `자동조정_허용`이 없어 **`auth` 0.20이 통째로 사라진다.**

남은 셋에 비율대로 나눠주지 않고, `auth`가 담당하던 "화주가 조정 권한을 넘겼는가"를
**`원화주_조정권한_증빙`**이 대신하게 한다. 의미가 가장 가깝고, v4에서 이 필드가
유연성 필드 전체를 잠그는 게이트로 실제 작동하기 때문이다.

| 항목 | v2 | v4 | 산식 |
|---|---:|---:|---|
| time | .40 | **.40** | 시간창_분 / 240 (상한 1.0) |
| veh | .15 | **.15** | 차량유연성 = 대체허용 |
| cargo | .25 | **.25** | 분할·동시적재·경유·순서변경 4개 평균 |
| auth | .20 | **.20** | 승인완료·원화주직접 1.0 / 승인대기 0.5 / 미승인 0 |

가중치를 유지했으므로 v2 산출값과 직접 비교할 수 있다.

In [5]:
yn = lambda c: (df[c].astype(str).str.strip() == "Y").astype(int)

df["flex_time"] = (df["시간창_분"] / 240).clip(upper=1.0)
df["flex_veh"] = (df["차량유연성"].astype(str).str.strip() == "대체허용").astype(int)
df["flex_cargo"] = (yn("분할운송") + yn("동시적재") + yn("경유허용") + yn("상하차_순서변경")) / 4
df["flex_auth"] = df["원화주_조정권한_증빙"].map(
    {"승인완료": 1.0, "승인대기": 0.5, "미승인": 0.0}).fillna(1.0)   # 결측 = 원화주직접

W = {"time": 0.40, "veh": 0.15, "cargo": 0.25, "auth": 0.20}
df["유연성지수"] = (W["time"] * df["flex_time"] + W["veh"] * df["flex_veh"]
                + W["cargo"] * df["flex_cargo"] + W["auth"] * df["flex_auth"])

# ⚠ 순환 참조 주의 — flex_auth는 등록주체가 '원화주직접'이면 정의상 1.0이다.
# 따라서 "등록주체별 유연성 차이"를 유연성지수로 검정하면 당연한 결과가 나온다.
# 그 비교에는 auth를 뺀 코어 지수를 쓴다(나머지 3항을 1.0으로 재정규화).
df["유연성지수_코어"] = (W["time"] * df["flex_time"] + W["veh"] * df["flex_veh"]
                    + W["cargo"] * df["flex_cargo"]) / (W["time"] + W["veh"] + W["cargo"])

## 1-3. 결과 파생변수 — v4에서 새로 가능해진 것

경매 결과가 있으므로 "조건 → 결과"를 직접 잇는 변수를 만든다.
Phase 4의 종속변수 3개가 전부 여기서 나온다.

- **`유찰`** — 조건이 나빠 아무도 안 잡은 건
- **`체결배율`** — 최종체결운임 / 기준운임. 조건이 운임을 얼마나 밀어올렸나
- **`초과운임`** — 체결 − 제시. 경매 라운드에서 인상된 절대액
- **`후보배율`** — 시간완화 / 현조건. 창만 열면 후보가 몇 배가 되나

In [6]:
df["유찰"] = (df["결과"] == "유찰").astype(int)
df["체결배율"] = df["최종체결운임"] / df["기준운임"]
df["초과운임"] = df["최종체결운임"] - df["제시운임"]
df["인상발생"] = (df["인상횟수"] > 0).astype(int)
df["후보배율"] = df["수락가능_시간완화"] / df["수락가능_현조건"].clip(lower=1)
df["긴급플래그"] = (df["긴급여부"] == "긴급").astype(int)
df["권한_미승인"] = (df["원화주_조정권한_증빙"].astype(str) == "미승인").astype(int)
df["개입수락"] = (df["개입적용"].fillna("").astype(str) != "").astype(int)

# 단가 — 거리로 정규화해야 노선 간 비교가 된다
df["기준_원per_km"] = df["기준운임"] / df["거리km"]
df["체결_원per_km"] = df["최종체결운임"] / df["거리km"]
df["중량_톤"] = df["중량_kg"] / 1000
df["적재율"] = df["중량_톤"] / df["톤급"]      # 톤급 대비 실중량 = 공간 낭비 지표

## 1-4. 데이터 품질 점검

v2에서 잡던 항목(허용창 선언≠실측, 자정 넘김) 중 상당수는 v4 구조상 발생할 수 없다.
대신 **v4에서만 검증 가능한 항목**을 넣는다 — 유찰건에 체결운임이 있으면 안 되고,
미승인 건은 시간창이 0이어야 하며(게이트), 인상이력의 마지막 값은 체결운임과 같아야 한다.

In [7]:
qc = {}
qc["행수"] = len(df)
qc["리드타임_음수"] = int((df["리드타임_h"] < 0).sum())
qc["시간창_음수"] = int((df["시간창_분"] < 0).sum())
qc["유찰인데_체결운임있음"] = int(((df["유찰"] == 1) & df["최종체결운임"].notna()).sum())
qc["성사인데_체결운임없음"] = int(((df["유찰"] == 0) & df["최종체결운임"].isna()).sum())
qc["체결<제시(역주행)"] = int((df["최종체결운임"] < df["제시운임"]).sum())
qc["미승인인데_유연성필드Y"] = int(((df["권한_미승인"] == 1) & (df["flex_cargo"] > 0)).sum())
qc["중량0이하"] = int((df["중량_kg"] <= 0).sum())
qc["적재율>1(과적)"] = int((df["적재율"] > 1).sum())
qc["긴급건수"] = int(df["긴급플래그"].sum())
qc["원화주_미승인"] = int(df["권한_미승인"].sum())
qc["개입수락_콜"] = int(df["개입수락"].sum())

# 인상이력 마지막 값 == 최종체결운임 인가 (개입으로 덮어쓴 건은 제외)
_h = df.loc[(df["인상이력"].fillna("") != "") & (df["개입수락"] == 0),
            ["인상이력", "최종체결운임"]].dropna()
_last = _h["인상이력"].astype(str).str.split("→").str[-1].astype(float)
qc["인상이력_말단≠체결"] = int((_last != _h["최종체결운임"]).sum())

# 구조적 결측 — 유찰건의 체결운임/배차시간은 삭제가 아니라 범주로 보존
df["체결_구분"] = np.where(df["최종체결운임"].isna(), "유찰", "체결")
qc["체결운임_결측"] = int(df["최종체결운임"].isna().sum())
qc["그중_유찰(구조적)"] = int((df["최종체결운임"].isna() & (df["유찰"] == 1)).sum())

_watch = ("유찰인데", "성사인데", "체결<", "미승인인데", "중량0", "인상이력_말단", "적재율")
print("\n[QC]")
for k, v in qc.items():
    print(f"  {k:24s} {v}{'   ← 확인 필요' if k.startswith(_watch) and v > 0 else ''}")


[QC]
  행수                       12000
  리드타임_음수                  0
  시간창_음수                   0
  유찰인데_체결운임있음              0
  성사인데_체결운임없음              0
  체결<제시(역주행)               0
  미승인인데_유연성필드Y             0
  중량0이하                    0
  적재율>1(과적)                0
  긴급건수                     1615
  원화주_미승인                  1511
  개입수락_콜                   1452
  인상이력_말단≠체결               0
  체결운임_결측                  1053
  그중_유찰(구조적)               1053


## 1-5. 저장

출처를 `_provenance.json`에 남긴다. Phase 4·5이 이걸 읽어 어떤 데이터로 적합했는지 표시한다.

In [8]:
df.to_csv(OUT_DIR / "phase1_분석테이블.csv", index=False, encoding="utf-8-sig")
prop.to_csv(OUT_DIR / "phase1_제안로그.csv", index=False, encoding="utf-8-sig")
pd.Series(qc).rename("건수").to_frame().to_csv(
    OUT_DIR / "phase1_품질점검.csv", encoding="utf-8-sig")

(OUT_DIR / "_provenance.json").write_text(json.dumps({
    "source": IN_XLSX.name, "rows": len(df), "cols": int(df.shape[1]),
    "제안로그": len(prop), "화주수": int(df["화주ID"].nunique()),
    "생성": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
}, ensure_ascii=False, indent=1), encoding="utf-8")

print("\n[유연성지수]")
print(df["유연성지수"].describe().round(3).to_string())
print("\n[구간별]")
print(df.groupby("구간").agg(콜=("콜ID", "size"), 유찰률=("유찰", "mean"),
                           체결배율=("체결배율", "mean")).round(3).to_string())
print(f"\n[저장] {OUT_DIR}")


[유연성지수]
count    12000.000
mean         0.343
std          0.207
min          0.000
25%          0.200
50%          0.325
75%          0.475
max          1.000

[구간별]
        콜    유찰률   체결배율
구간                     
개입전  2404  0.099  1.148
개입후  9596  0.085  1.146

[저장] /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out
